Environment setup and imports

In [1]:
# Python + DS stack
# !pip install pandas numpy plotly tensorflow scikit-learn
import os
import math
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime, timedelta
from typing import Dict, Tuple

# 🚀 FidZulu Notebook Bootstrap
import sys
import importlib, fidzulu
importlib.reload(fidzulu)
import pandas as pd
pd.options.plotting.backend = "plotly"
import plotly.graph_objects as go

import pandas as pd
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers # type: ignore

# FidZulu imports: FidZulu bootstrap + TensorFlow version check
# 🚀 FidZulu Notebook Bootstrap

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
src_path = os.path.join(project_root, "src")

# Add project root
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Add src/
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from fidzulu.db import oracle_engine
from fidzulu.repositories.price_repository import PriceRepository
from fidzulu.business.price_wrangler import PriceDataWrangler
print(tf.__version__)
# This forces TensorFlow to compile your model into a fast graph
tf.config.run_functions_eagerly(False)


2.21.0


GPU vs CPU

In [2]:
# List available devices
print("Available devices:", tf.config.list_physical_devices())

# For small tabular models, GPU overhead is larger than GPU benefit
# Thus, force TensorFlow to use CPU only
tf.config.set_visible_devices([], 'GPU')

# Or, if you want to force GPU (assuming you have one):
# gpus = tf.config.list_physical_devices('GPU')
# if gpus:
#     tf.config.set_visible_devices(gpus[0], 'GPU')

print("Now using:", tf.config.get_visible_devices())

Available devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
Now using: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


## Load Raw Data

We connect to the database and retrieve raw price data for a given category.

In [3]:
# Adjust connection string to your environment
engine = oracle_engine()
repo = PriceRepository(engine)
raw_data = repo.get_prices_by_category(cat_id=13) # Vegetable category
print(type(raw_data))
print(raw_data.keys())

raw_data

2026-04-09 11:32:15,157 INFO fidzulu.config: Loaded DBConfig: host=localhost, port=1521, service=xepdb1
2026-04-09 11:32:15,158 INFO fidzulu.db: {'event': 'engine_create_attempt', 'db_user': 'fidzulu_pythonmluser', 'host': 'localhost', 'port': 1521, 'service_name': 'xepdb1'}
2026-04-09 11:32:15,159 INFO fidzulu.db: Creating Oracle engine with DSN (password redacted from log)
2026-04-09 11:32:16,076 INFO fidzulu.db: Oracle engine created successfully
2026-04-09 11:32:16,078 INFO fidzulu.repositories.price_repository: {'event': 'query_attempt', 'operation': 'get_prices_by_category', 'query': 'prices_by_category', 'params': {'cat_id': 13}}
2026-04-09 11:32:18,392 WARNING fidzulu.repositories.price_repository: {'event': 'anomaly', 'message': 'filtered_invalid_price_rows', 'details': {'category': 13, 'skipped_rows': 1}}


<class 'dict'>
dict_keys(['CategoryID', 105, 106])


{'CategoryID': 13,
 105: {'prices': [9.95,
   2.5,
   2.65,
   2.4,
   2.75,
   2.55,
   2.85,
   2.6,
   2.95,
   2.7,
   3.05,
   2.8,
   3.15,
   2.95,
   3.3,
   3.05,
   3.4,
   3.15,
   3.5,
   3.25,
   3.6,
   3.35,
   10.15,
   3.45,
   200.0,
   3.8,
   3.55],
  'start_dates': [datetime.datetime(2022, 11, 1, 0, 0),
   datetime.datetime(2023, 1, 1, 0, 0),
   datetime.datetime(2023, 2, 1, 0, 0),
   datetime.datetime(2023, 3, 1, 0, 0),
   datetime.datetime(2023, 4, 1, 0, 0),
   datetime.datetime(2023, 5, 1, 0, 0),
   datetime.datetime(2023, 6, 1, 0, 0),
   datetime.datetime(2023, 7, 1, 0, 0),
   datetime.datetime(2023, 8, 1, 0, 0),
   datetime.datetime(2023, 9, 1, 0, 0),
   datetime.datetime(2023, 10, 1, 0, 0),
   datetime.datetime(2023, 11, 1, 0, 0),
   datetime.datetime(2023, 12, 1, 0, 0),
   datetime.datetime(2024, 1, 1, 0, 0),
   datetime.datetime(2024, 2, 1, 0, 0),
   datetime.datetime(2024, 3, 1, 0, 0),
   datetime.datetime(2024, 4, 1, 0, 0),
   datetime.datetime(2024, 5, 1